In [75]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
import pandas as pd
import glob
import numpy as np
import joblib
import pickle
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization, GlobalMaxPooling1D, Conv1D, SpatialDropout1D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [77]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 8236
Dari 1 file: ['df_merge.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8236 entries, 0 to 8235
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8236 non-null   float64
 1   title             8236 non-null   object 
 2   search_role       8236 non-null   object 
 3   job_level         8236 non-null   object 
 4   company           8236 non-null   object 
 5   location          5902 non-null   object 
 6   salary_avg        1667 non-null   float64
 7   extracted_skills  8236 non-null   object 
 8   skills_count      8236 non-null   int64  
 9   job_url           8236 non-null   object 
 10  scraped_at        8236 non-null   object 
 11  job_description   8231 non-null   object 
dtypes: float64(2), int64(1), object(9)
memory usage: 772.3+ KB


In [78]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg', 'company', 'location', 'job_url', 'search_location', 'scraped_at'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
df.dropna(inplace=True)
print(df.isnull().sum())

Total duplicated rows: 155
id                  0
title               0
search_role         0
extracted_skills    0
skills_count        0
job_description     0
dtype: int64


In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8231 entries, 0 to 8235
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                8231 non-null   float64
 1   title             8231 non-null   object 
 2   search_role       8231 non-null   object 
 3   extracted_skills  8231 non-null   object 
 4   skills_count      8231 non-null   int64  
 5   job_description   8231 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 450.1+ KB


In [80]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

df['text_input'] = df['title'] + ' ' + df['extracted_skills_clean']
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

In [81]:
min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Backend Developer      1374
Fullstack Developer    1279
DevOps Engineer        1254
Data Analyst           1088
Frontend Developer     1024
Data Engineer          1005
Software Engineer       492
ML Engineer             399
Data Scientist          316
Name: count, dtype: int64


In [82]:
df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer', 'Software Developer']
df = df[~df['search_role'].isin(roles_to_drop)]

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Backend Developer      1374
Fullstack Developer    1279
DevOps Engineer        1254
Data Analyst           1088
Frontend Developer     1024
Data Engineer          1005
ML Engineer             399
Data Scientist          316
Name: count, dtype: int64


In [83]:
def map_roles(role):
    if role == 'Fullstack Developer':
        return ['Frontend Developer', 'Backend Developer']
    else:
        return [role]

df['role_list'] = df['search_role'].apply(map_roles)

mlb = MultiLabelBinarizer()
y_multilabel = mlb.fit_transform(df['role_list'])

df['label'] = list(y_multilabel)
classes = mlb.classes_
num_classes = len(classes)

print(f"Classes: {classes}")
print(f"Num classes: {num_classes}")

Classes: ['Backend Developer' 'Data Analyst' 'Data Engineer' 'Data Scientist'
 'DevOps Engineer' 'Frontend Developer' 'ML Engineer']
Num classes: 7


In [84]:
stop_words_en = set(stopwords.words('english'))
stop_words_id = set(stopwords.words('indonesian'))

all_stop_words = stop_words_en.union(stop_words_id)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = text.split()
    clean_words = [w for w in words if w not in all_stop_words]

    return ' '.join(clean_words)

df['text_input'] = df['text_input'].apply(clean_text)

In [85]:
print("=== Label alignment check ===")
for i in range(5):
    print(f"Role: {df['search_role'].iloc[i]}")
    print(f"Label: {df['label'].iloc[i]}")
    print(f"Title: {df['title'].iloc[i]}")
    print()

=== Label alignment check ===
Role: ML Engineer
Label: [0 0 0 0 0 0 1]
Title: AI & Automation Engineer

Role: ML Engineer
Label: [0 0 0 0 0 0 1]
Title: AI Officer (Part-Time/Project-Based)

Role: ML Engineer
Label: [0 0 0 0 0 0 1]
Title: Engineering Staff

Role: ML Engineer
Label: [0 0 0 0 0 0 1]
Title: Data Engineer

Role: ML Engineer
Label: [0 0 0 0 0 0 1]
Title: Programmable Logic Controller (PLC) Automation Engineer



In [86]:
X_train, X_val, df_y_train, df_y_val = train_test_split(
    df['text_input'], df[['search_role', 'label']],
    test_size=0.2, random_state=42, stratify=df['search_role']
)

X_train_bal = X_train
y_train_bal = np.stack(df_y_train['label'].values)
y_val_arr = np.stack(df_y_val['label'].values)

print(f"\nTotal training samples: {len(X_train_bal)}")
print(f"Total val samples: {len(X_val)}")

vectorizer = TextVectorization(max_tokens=3000, output_mode='int', output_sequence_length=300, ngrams=2)
vectorizer.adapt(X_train_bal.to_numpy())


Total training samples: 6191
Total val samples: 1548


In [87]:
print("=== Sample text inputs ===")
for role in df['search_role'].unique():
    sample = df[df['search_role'] == role]['text_input'].iloc[0]
    print(f"\n[{role}]")
    print(sample[:200])

print("\n=== Vectorizer vocab sample ===")
print(vectorizer.get_vocabulary()[:50])
print(f"\nVocab size: {len(vectorizer.get_vocabulary())}")

=== Sample text inputs ===

[ML Engineer]
ai automation engineer python java javascript

[Backend Developer]
coding teacher elementary middle school part time

[Data Analyst]
commercial superintendent

[Data Engineer]
admin support staff power bi

[Data Scientist]
data analyst admin staff

[DevOps Engineer]
sap hana technical architect scala azure git agile aws

[Frontend Developer]
front end developer talent pool supervisor react java javascript

[Fullstack Developer]
golang developer golang redis kubernetes postgresql sql kafka scala docker mongodb

=== Vectorizer vocab sample ===
['', '[UNK]', np.str_('sql'), np.str_('engineer'), np.str_('git'), np.str_('java'), np.str_('developer'), np.str_('python'), np.str_('analyst'), np.str_('data'), np.str_('scala'), np.str_('javascript'), np.str_('react'), np.str_('postgresql'), np.str_('agile'), np.str_('aws'), np.str_('mysql'), np.str_('docker'), np.str_('senior'), np.str_('php'), np.str_('azure'), np.str_('manager'), np.str_('gcp'), np.str

In [88]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [89]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier # Tambahan krusial untuk Multi-label

# 1. Transformasi teks jadi angka
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_train_tfidf = tfidf.fit_transform(X_train_bal)
X_val_tfidf = tfidf.transform(X_val)

print(f"Dimensi TF-IDF: {X_train_tfidf.shape}\n")

# 2. Model 1: Logistic Regression (Versi Multi-label)
print("=== HASIL LOGISTIC REGRESSION ===")
base_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model = OneVsRestClassifier(base_lr) # Bungkus model di sini
lr_model.fit(X_train_tfidf, y_train_bal)
y_pred_lr = lr_model.predict(X_val_tfidf)
print(classification_report(y_val_arr, y_pred_lr, target_names=classes))

# 3. Model 2: Multinomial Naive Bayes (Versi Multi-label)
print("\n=== HASIL MULTINOMIAL NAIVE BAYES ===")
base_nb = MultinomialNB()
nb_model = OneVsRestClassifier(base_nb) # Bungkus model di sini
nb_model.fit(X_train_tfidf, y_train_bal)
y_pred_nb = nb_model.predict(X_val_tfidf)
print(classification_report(y_val_arr, y_pred_nb, target_names=classes))

Dimensi TF-IDF: (6191, 5000)

=== HASIL LOGISTIC REGRESSION ===
                    precision    recall  f1-score   support

 Backend Developer       0.57      0.78      0.66       531
      Data Analyst       0.52      0.88      0.66       217
     Data Engineer       0.54      0.82      0.65       201
    Data Scientist       0.40      0.81      0.54        63
   DevOps Engineer       0.50      0.84      0.62       251
Frontend Developer       0.54      0.76      0.63       461
       ML Engineer       0.34      0.61      0.44        80

         micro avg       0.52      0.79      0.63      1804
         macro avg       0.49      0.79      0.60      1804
      weighted avg       0.53      0.79      0.63      1804
       samples avg       0.56      0.80      0.64      1804


=== HASIL MULTINOMIAL NAIVE BAYES ===
                    precision    recall  f1-score   support

 Backend Developer       0.61      0.59      0.60       531
      Data Analyst       0.75      0.41      0.53    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [90]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = SpatialDropout1D(0.3)(x)
x = Conv1D(128, 5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dropout(0.5)(x)

outputs = Dense(num_classes, activation='sigmoid', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['binary_accuracy'])

history = model.fit(
    X_train_bal.to_numpy(), y_train_bal,
    validation_data=(X_val.to_numpy(), y_val_arr),
    epochs=40,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - binary_accuracy: 0.8129 - loss: 0.5216 - val_binary_accuracy: 0.8335 - val_loss: 0.4262 - learning_rate: 0.0010
Epoch 2/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - binary_accuracy: 0.8365 - loss: 0.4076 - val_binary_accuracy: 0.8475 - val_loss: 0.3515 - learning_rate: 0.0010
Epoch 3/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - binary_accuracy: 0.8454 - loss: 0.3533 - val_binary_accuracy: 0.8526 - val_loss: 0.3228 - learning_rate: 0.0010
Epoch 4/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - binary_accuracy: 0.8527 - loss: 0.3245 - val_binary_accuracy: 0.8583 - val_loss: 0.3104 - learning_rate: 0.0010
Epoch 5/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - binary_accuracy: 0.8614 - loss: 0.3041 - val_binary_accuracy: 0.8635 - val_loss: 0.3021 - learning_rate: 0.0010
Epoch 6/40
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - binary_accuracy: 0.8711 - loss: 0.2852 - val_binary_accuracy: 0.8654 - val_loss: 0.2951 - learning_rate: 0.0010
Epoch 7/40

In [91]:
y_pred_probs = model.predict(X_val.to_numpy())
y_pred = (y_pred_probs > 0.5).astype(int)

print(classification_report(y_val_arr, y_pred, target_names=classes))

49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
                    precision    recall  f1-score   support

 Backend Developer       0.58      0.71      0.64       531
      Data Analyst       0.65      0.61      0.63       217
     Data Engineer       0.66      0.59      0.62       201
    Data Scientist       0.79      0.30      0.44        63
   DevOps Engineer       0.64      0.54      0.59       251
Frontend Developer       0.55      0.53      0.54       461
       ML Engineer       0.78      0.31      0.45        80

         micro avg       0.60      0.58      0.59      1804
         macro avg       0.66      0.51      0.56      1804
      weighted avg       0.61      0.58      0.59      1804
       samples avg       0.51      0.58      0.53      1804



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [92]:
model.save('job_role_model.keras')
joblib.dump(mlb, 'label_encoder.pkl')

loaded_model = tf.keras.models.load_model('job_role_model.keras')
loaded_mlb = joblib.load('label_encoder.pkl')

def rekomendasi_role(skills_text, threshold=0.5):
    input_tensor = tf.constant([skills_text], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)[0]

    active_indices = np.where(pred_probs > threshold)[0]

    if len(active_indices) == 0:
        best_idx = np.argmax(pred_probs)
        return [loaded_mlb.classes_[best_idx]], [pred_probs[best_idx]]

    predicted_roles = loaded_mlb.classes_[active_indices]
    confidences = pred_probs[active_indices]

    return predicted_roles, confidences

skill_input = "experience building REST APIs Laravel FastAPI managing databases PostgreSQL"
roles, confs = rekomendasi_role(skill_input)

print(f"\n--- Hasil Inference ---")
print(f"Skill Input : {skill_input}")
for r, c in zip(roles, confs):
    print(f"Cocok untuk : {r} ({c:.1%})")


--- Hasil Inference ---
Skill Input : experience building REST APIs Laravel FastAPI managing databases PostgreSQL
Cocok untuk : Backend Developer (67.6%)
Cocok untuk : Frontend Developer (63.2%)
